In [1]:
!pip install paddlepaddle
!pip install paddleocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -U langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.8/495.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.7
    Uninstalling langchain-core-1.2.7:
      Successfully uninstalled langchain-core-1.2.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is inco

In [1]:
import os
# Must be before 'import paddle'
os.environ['FLAGS_enable_pir_api'] = '0'

import paddle
import sys
from types import ModuleType

# 1. LangChain Compatibility Shim
try:
    from langchain_core.documents import Document
    m = ModuleType("langchain.docstore.document")
    m.Document = Document
    sys.modules["langchain.docstore.document"] = m
    import langchain_text_splitters as ts
    sys.modules["langchain.text_splitter"] = ts
except ImportError:
    pass

# 2. OCR Initialization
from paddleocr import PaddleOCR

# The key here is enable_mkldnn=False
ocr = PaddleOCR(use_angle_cls=True, lang='en', enable_mkldnn=False)

# 3. Test Run
#img_path = '/content/WhatsApp Image 2026-01-28 at 18.16.19.jpeg' # Make sure this path is correct!
#result = ocr.ocr(img_path)

#for idx in range(len(result)):
#    for line in result[idx]:
#        print(line[1][0]) # This prints the detected text

ModuleNotFoundError: No module named 'paddle'

In [3]:
def run_ocr(path: str):
  result = ocr.predict(path)

  text_doc = ''
  for res in result:
    for text_line in res['rec_texts']:
      text_doc += (text_line + '\n')

  print("--- RAW EXTRACTED TEXT ---")
  print(text_doc)

In [4]:
run_ocr('/content/WhatsApp Image 2026-01-28 at 18.16.19.jpeg')

NameError: name 'ocr' is not defined

In [ ]:
def smart_layout_aggregation(ocr_result, y_tolerance=15):
    """
    Aggregates scattered OCR boxes into visual lines based on Y-coordinates.
    Args:
        ocr_result: The raw output from ocr.predict()
        y_tolerance: Pixels of vertical difference to consider items on the same line
    """
    extracted_items = []

    # 1. Flatten the data into a list of dicts with coordinates
    for page in ocr_result:
        if 'rec_texts' in page and 'dt_polys' in page:
            texts = page['rec_texts']
            boxes = page['dt_polys']

            for text, box in zip(texts, boxes):
                # Calculate the top-left corner (y_min, x_min)
                # box format is [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
                y_min = min(point[1] for point in box)
                x_min = min(point[0] for point in box)

                extracted_items.append({
                    "text": text,
                    "y": y_min,
                    "x": x_min
                })

    # 2. Sort primarily by Y (vertical), secondarily by X (horizontal)
    extracted_items.sort(key=lambda k: (k['y'], k['x']))

    # 3. Group items into "Visual Lines"
    lines = []
    current_line = []
    previous_y = -100

    for item in extracted_items:
        # If the Y difference is small, it's the same line (e.g., columns)
        if abs(item['y'] - previous_y) < y_tolerance:
            current_line.append(item)
        else:
            # New line detected
            if current_line:
                # Sort the previous line strictly Left-to-Right
                current_line.sort(key=lambda k: k['x'])
                lines.append(current_line)

            current_line = [item]
            previous_y = item['y']

    # Append the final line
    if current_line:
        current_line.sort(key=lambda k: k['x'])
        lines.append(current_line)

    # 4. Construct the Final Context String
    # We join items in a line with " | " to represent visual separation to the LLM
    final_text = ""
    for line in lines:
        line_text = " | ".join([item['text'] for item in line])
        final_text += line_text + "\n"

    return final_text

In [ ]:
print(smart_layout_aggregation(ocr.predict('/content/WhatsApp Image 2026-01-28 at 18.16.19.jpeg')))

A spokesperson for the firm at press briefings and interviews | epresents nigh-levet corpolate clienics inr court
 
advising clients accordingly
Networking at industry events and conferences to build a client base
behalf of the firm
Act as the legal expert and present opinions based on legal analysis
regarding case issues to junior Associate Attorneys
Responsible for hiring junior associate attorneys, allocating work
to them and managing support staff such as legal secretaries and
paralegals
Develop strategies regarding depositions, pre-trial hearings, and trial
hearings
Negotiate in settlements of legal disputes
Associate Attorney Il at California Media Consultants, Orange County
January 2016- November 2017
Compiled and prepared more than 1500 pre-trial briefs and over 800 000
pages of tax records during the last five years.
Represented media clients in copyright infringement actions and
proceedings
Draft tolling agreements, pleadings,initial disclosures, 26( reports,
Compile response